# CAVR on Colab (A100)

End-to-end walkthrough: install -> verify -> collect demos -> train -> evaluate -> baselines -> ablation.

**Before running:** `Runtime -> Change runtime type -> GPU -> A100`.

Persist results to Google Drive by running the Drive-mount cell.

## 1. Mount Drive (optional but recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put checkpoints + demos under Drive so they survive runtime restarts.
import os
os.makedirs('/content/drive/MyDrive/cavr', exist_ok=True)
!ln -sfn /content/drive/MyDrive/cavr /content/cavr_persist

## 2. Clone the repo and bootstrap

In [ ]:
# EDIT THIS: your fork/repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/r3m-4-DLRM.git'
!rm -rf /content/cavr && git clone $REPO_URL /content/cavr
!bash /content/cavr/colab/bootstrap.sh /content/cavr

In [ ]:
# Activate rendering backend in the live Python process too.
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
%cd /content/cavr

## 3. Verify the install

In [ ]:
!python scripts/verify_setup.py

## 4. Collect demonstrations

Headless scripted collector. For persistence, point `--save-dir` at the Drive mount.

In [ ]:
!python scripts/collect_demos.py --env PickPlace --num-demos 50 \
    --save-dir /content/cavr_persist/demos_pickplace
!python scripts/collect_demos.py --env Door --num-demos 50 \
    --save-dir /content/cavr_persist/demos_door

## 5. Smoke test (2 epochs, small batch)

In [ ]:
!python scripts/train.py --model cavr --env PickPlace \
    --data-dir /content/cavr_persist/demos_pickplace \
    --epochs 2 --batch-size 16

## 6. Full training run on A100

In [ ]:
!python scripts/train.py --model cavr --env PickPlace \
    --data-dir /content/cavr_persist/demos_pickplace \
    --epochs 200 --batch-size 64

### Push checkpoints to Drive

In [ ]:
!rsync -a --delete checkpoints/ /content/cavr_persist/checkpoints/

## 7. Evaluate

In [ ]:
!python scripts/evaluate.py --model cavr --env PickPlace \
    --checkpoint checkpoints/cavr_PickPlace/best.pt \
    --num-episodes 50

## 8. Baselines sweep (CAVR vs R3M vs VC-1)

In [ ]:
!python scripts/run_baselines.py --env PickPlace \
    --data-dir /content/cavr_persist/demos_pickplace

## 9. Ablation sweep (ViT-L/B x masked/unmasked)

In [ ]:
!python scripts/run_ablation.py --env PickPlace \
    --data-dir /content/cavr_persist/demos_pickplace

In [ ]:
!rsync -a outputs/ /content/cavr_persist/outputs/